<a href="https://colab.research.google.com/github/CarlosMohr/Sistemas-de-Informa-o-Gerenciais-2026.2/blob/main/escola_evasao_tratamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd



bruto = pd.read_csv('/content/matriculas_escola.csv')
print(bruto.shape)
bruto.head()

(384, 8)


,matricula_id,aluno,responsavel,curso,turno,situacao,mensalidade,data_matricula
0,5001,Nathalia Kunz,Gabriela Fagundes,Técnico em Segurança do Trabalho,Manhã,Concluído,310.0,2026-02-03
1,5002,Sabrina Brum,Marcos Carvalho,Técnico em Contabilidade,Tarde,CONCLUIDO,310.0,2025-02-20
2,5003,Tiago Schneider,Nathalia Machado Rocha,Técnico em Administração,Noite,EVADIDO,340.0,2026-04-23
3,5004,Elisa Lima,Nelson Fontoura,Técnico em Administração,Noite,CURSANDO,330.0,2025-02-13
4,5005,Alexandre Brum,Leonardo Brum,Enfermagem,Noite,CONCLUIDO,490.0,2025-03-18


In [ ]:
print("Linhas x colunas:", bruto.shape)
print("\nTipos lidos pelo pandas:")
print(bruto.dtypes)
print("\nAusentes por coluna:")
print(bruto.isna().sum())
print("\nGrafias distintas onde deveria haver poucas:")
for col in ['curso', 'turno', 'situacao']:
      print(f"  {col}: {bruto[col].nunique()} grafias distintas")

Linhas x colunas: (384, 8)

Tipos lidos pelo pandas:
matricula_id        int64
aluno              object
responsavel        object
curso              object
turno              object
situacao           object
mensalidade       float64
data_matricula     object
dtype: object

Ausentes por coluna:
matricula_id       0
aluno              0
responsavel       18
curso              0
turno              0
situacao           0
mensalidade        0
data_matricula     0
dtype: int64

Grafias distintas onde deveria haver poucas:
  curso: 28 grafias distintas
  turno: 3 grafias distintas
  situacao: 6 grafias distintas


In [ ]:
for col in ['curso', 'situacao', 'turno']:
      print(f"--- {col} ---")
      print(bruto[col].value_counts(dropna=False))
      print()

--- curso ---
curso
Técnico em Administração             72
Técnico em Informática               44
Técnico em Segurança do Trabalho     38
TECNICO EM ADMINISTRACAO             32
Técnico em Enfermagem                30
Técnico em Estética                  24
Técnico em Logística                 23
Técnico em Contabilidade             20
SEGURANCA DO TRABALHO                14
TECNICO EM INFORMATICA               14
Enfermagem                           14
TECNICO EM CONTABILIDADE             11
TECNICO EM LOGISTICA                 10
TECNICO EM ESTETICA                   8
TECNICO EM ENFERMAGEM                 6
TECNICO EM ADMINISTRACAO              5
Técnico em Administração              3
Técnico em Informática                3
Técnico em Logística                  2
TECNICO EM LOGISTICA                  2
Técnico em Enfermagem                 2
Enfermagem                            1
TECNICO EM ENFERMAGEM                 1
Técnico em Segurança do Trabalho      1
Técnico em Contabili

In [ ]:
df = bruto.copy()

In [ ]:
antes = df['aluno'].nunique()
df['aluno'] = df['aluno'].str.strip()
print(f"Alunos distintos: {antes} -> {df['aluno'].nunique()}")

Alunos distintos: 362 -> 344


In [ ]:
import unicodedata
def normaliza(serie):
    """Maiúsculas, sem acento, sem espaço sobrando nem espaço duplicado."""
    return (serie.astype('string')
    .str.strip()
    .str.upper()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('ascii')
    .str.replace(r'\s+', ' ', regex=True))

In [ ]:
mapa_curso = {    'TECNICO EM ADMINISTRACAO':        'TÉCNICO EM ADMINISTRAÇÃO',    'ADMINISTRACAO':                   'TÉCNICO EM ADMINISTRAÇÃO',    'TECNICO EM INFORMATICA':          'TÉCNICO EM INFORMÁTICA',    'INFORMATICA':                     'TÉCNICO EM INFORMÁTICA',    'TECNICO EM ENFERMAGEM':           'TÉCNICO EM ENFERMAGEM',    'ENFERMAGEM':                      'TÉCNICO EM ENFERMAGEM',    'TECNICO EM SEGURANCA DO TRABALHO':'TÉCNICO EM SEGURANÇA DO TRABALHO',    'SEGURANCA DO TRABALHO':           'TÉCNICO EM SEGURANÇA DO TRABALHO',    'TECNICO EM LOGISTICA':            'TÉCNICO EM LOGÍSTICA',    'LOGISTICA':                       'TÉCNICO EM LOGÍSTICA',    'TECNICO EM ESTETICA':             'TÉCNICO EM ESTÉTICA',    'ESTETICA':                        'TÉCNICO EM ESTÉTICA',    'TECNICO EM CONTABILIDADE':        'TÉCNICO EM CONTABILIDADE',    'CONTABILIDADE':                   'TÉCNICO EM CONTABILIDADE',}
curso_norm = normaliza(df['curso'])
sem_destino = sorted(set(curso_norm.dropna()) - set(mapa_curso))
print("Grafias sem destino no dicionário:", sem_destino if sem_destino else "nenhuma")
# se aparecer alguma, acrescente a linha correspondente no mapa_curso acima e rode de novo
df['curso'] = curso_norm.map(mapa_curso)
print("Ausentes depois do map():", df['curso'].isna().sum())
print(f"Cursos distintos: {df['curso'].nunique()}\n")
print(df['curso'].value_counts())

Grafias sem destino no dicionário: nenhuma
Ausentes depois do map(): 0
Cursos distintos: 7

curso
TÉCNICO EM ADMINISTRAÇÃO            112
TÉCNICO EM INFORMÁTICA               62
TÉCNICO EM ENFERMAGEM                54
TÉCNICO EM SEGURANÇA DO TRABALHO     53
TÉCNICO EM LOGÍSTICA                 37
TÉCNICO EM ESTÉTICA                  34
TÉCNICO EM CONTABILIDADE             32
Name: count, dtype: int64


### 2.4 `situacao` — 4 grafias em 3 situaçõesPor quê: se "CONCLUIDO" e "CONCLUÍDO" ficarem separados, o denominador da taxa deevasão muda conforme a secretária que digitou. Depois de normalizar, as 4 grafiasviram 3 chaves e o dicionário só reintroduz o acento oficial.

In [ ]:
mapa_situacao = {'CURSANDO': 'CURSANDO', 'CONCLUIDO': 'CONCLUÍDO', 'EVADIDO': 'EVADIDO'}situacao_norm = normaliza(df['situacao'])sem_destino = sorted(set(situacao_norm.dropna()) - set(mapa_situacao))print("Grafias sem destino no dicionário:", sem_destino if sem_destino else "nenhuma")df['situacao'] = situacao_norm.map(mapa_situacao)print("Ausentes depois do map():", df['situacao'].isna().sum())print(df['situacao'].value_counts())

### 2.5 `turno`Mesma lógica.

In [ ]:
mapa_turno = {'MANHA': 'MANHÃ', 'TARDE': 'TARDE', 'NOITE': 'NOITE'}turno_norm = normaliza(df['turno'])sem_destino = sorted(set(turno_norm.dropna()) - set(mapa_turno))print("Grafias sem destino no dicionário:", sem_destino if sem_destino else "nenhuma")df['turno'] = turno_norm.map(mapa_turno)print("Ausentes depois do map():", df['turno'].isna().sum())print(df['turno'].value_counts())

### 2.6 Tipos — data como data, número como númeroO README diz que as demais colunas já vêm limpas, mas o checklist cobra os tipos.Converter é barato e o `errors='coerce'` mostra na hora se algum valor não é o queparece.

In [ ]:
# se as datas estiverem em dd/mm/aaaa, acrescente dayfirst=Truedf['data_matricula'] = pd.to_datetime(df['data_matricula'], errors='coerce')df['mensalidade'] = pd.to_numeric(df['mensalidade'], errors='coerce')print("Datas que não converteram:", df['data_matricula'].isna().sum())print("Mensalidades que não converteram:", df['mensalidade'].isna().sum())print("\nPeríodo coberto:", df['data_matricula'].min().date(), "a", df['data_matricula'].max().date())print()print(df.dtypes)

### 2.7 Duplicatas — rematrícula digitada duas vezesDefinição adotada: mesma pessoa + mesmo curso + mesma data de matrícula = umadigitação repetida, não uma matrícula nova.Qual cópia fica: a de **menor `matricula_id`**, ou seja, o primeiro registrolançado. As colunas restantes são idênticas nas duplicatas, então a escolha nãomuda nenhum número — mas fica documentada para o resultado ser reproduzível.

In [ ]:
chave = ['aluno', 'curso', 'data_matricula']antes = len(df)df = df.sort_values('matricula_id').drop_duplicates(subset=chave, keep='first')print(f"Linhas: {antes} -> {len(df)}  ({antes - len(df)} rematrículas digitadas duas vezes)")

### 2.8 Validação — a base passou?

In [ ]:
assert df['curso'].isna().sum() == 0,    "sobrou curso sem mapear"assert df['situacao'].isna().sum() == 0, "sobrou situação sem mapear"assert df['turno'].isna().sum() == 0,    "sobrou turno sem mapear"assert df['curso'].nunique() == 7,       f"esperado 7 cursos, achei {df['curso'].nunique()}"assert df['situacao'].nunique() == 3,    f"esperado 3 situações, achei {df['situacao'].nunique()}"assert df['turno'].nunique() == 3,       f"esperado 3 turnos, achei {df['turno'].nunique()}"assert df.duplicated(subset=chave).sum() == 0, "ainda há duplicatas"assert df['matricula_id'].is_unique,     "matricula_id repetido"print("Base tratada:", df.shape)print("\nAusentes por coluna:")print(df.isna().sum())

In [ ]:
df.to_csv('matriculas_escola_tratado.csv', index=False)print("Salvo como matriculas_escola_tratado.csv — o arquivo original não foi tocado.")

---## 3. As três respostasTodas calculadas sobre a base tratada. Cada número vem com o recorte: sobrequantas linhas ele foi calculado.

### 3.1 Qual é a taxa de evasão da escola? E por curso?Aqui há uma escolha de denominador que muda a resposta, e ela é o assunto daregra documentada na seção 4. Calculo as duas versões.

In [ ]:
n = len(df)evadidos   = (df['situacao'] == 'EVADIDO').sum()encerradas = df['situacao'].isin(['EVADIDO', 'CONCLUÍDO']).sum()print(f"Base tratada: {n} matrículas")print(f"  evadidos:  {evadidos}")print(f"  concluído: {(df['situacao'] == 'CONCLUÍDO').sum()}")print(f"  cursando:  {(df['situacao'] == 'CURSANDO').sum()}")print()print(f"Taxa sobre todas as matrículas:     {evadidos/n:.1%}  ({evadidos}/{n})")print(f"Taxa sobre matrículas encerradas:   {evadidos/encerradas:.1%}  ({evadidos}/{encerradas})")

In [ ]:
def tabela_taxa(coluna, apenas_encerradas=False):    base = df[df['situacao'].isin(['EVADIDO', 'CONCLUÍDO'])] if apenas_encerradas else df    t = (base.assign(evadiu=base['situacao'].eq('EVADIDO'))             .groupby(coluna)             .agg(matriculas=('evadiu', 'size'), evasoes=('evadiu', 'sum')))    t['taxa_%'] = (t['evasoes'] / t['matriculas'] * 100).round(1)    return t.sort_values('taxa_%', ascending=False)print("TAXA DE EVASÃO POR CURSO — sobre todas as matrículas")print(tabela_taxa('curso'))print()print("TAXA DE EVASÃO POR CURSO — só matrículas encerradas")print(tabela_taxa('curso', apenas_encerradas=True))

Compare a coluna `evasoes` com a coluna `taxa_%`: o curso com mais evasões emnúmero absoluto não é o com a pior taxa. A pergunta é sobre a taxa.

### 3.2 Qual turno concentra a evasão?

In [ ]:
print("TAXA DE EVASÃO POR TURNO — sobre todas as matrículas")print(tabela_taxa('turno'))print()print("TAXA DE EVASÃO POR TURNO — só matrículas encerradas")print(tabela_taxa('turno', apenas_encerradas=True))

In [ ]:
# A escola tem mais aula à noite. Isso é volume ou é risco?distribuicao = df['turno'].value_counts(normalize=True).mul(100).round(1)print("Participação de cada turno no total de matrículas (%):")print(distribuicao)print()print("Participação de cada turno no total de evasões (%):")print(df[df['situacao'] == 'EVADIDO']['turno'].value_counts(normalize=True).mul(100).round(1))

Se a fatia do turno nas evasões for maior que a fatia dele nas matrículas, o turnoconcentra evasão acima do que o tamanho dele explicaria.

### 3.3 Quantos alunos distintos a escola atendeu?

In [ ]:
print("Matrículas na base tratada:", len(df))print("Alunos distintos:", df['aluno'].nunique())multiplas = df['aluno'].value_counts()print("Alunos com mais de uma matrícula:", (multiplas > 1).sum())print("\nExemplos (aluno em mais de um curso, o que é matrícula legítima):")print(multiplas[multiplas > 1].head())

---## 4. Uma regra documentada**Decisão mais discutível:** qual é o denominador da taxa de evasão.Quem ainda está CURSANDO não teve a chance de evadir — incluir essas pessoas nodenominador dilui a taxa e faz a escola parecer melhor do que é. Por outro lado, ataxa sobre matrículas encerradas ignora quem vai evadir no próximo semestre etende a superestimar. As duas versões estão calculadas acima; a regra abaixo é aque a escola deveria adotar como oficial.| Campo | Conteúdo ||---|---|| **Nome** | Taxa de Evasão por Coorte Encerrada || **Definição** | Percentual de matrículas já encerradas — concluídas ou evadidas — que terminaram em evasão. Matrículas com situação CURSANDO ficam de fora do cálculo porque o desfecho delas ainda não é conhecido. || **Fórmula** | `evadidos / (evadidos + concluídos) × 100` || **Recorte** | Base tratada, matrículas dos três semestres exportados || **Dono** | [nome do responsável — secretaria acadêmica ou coordenação] || **Vale desde** | [data] |Reporte sempre o número junto do recorte: "X% sobre N matrículas encerradas".Uma taxa sem denominador não é informação, conforme a Aula 02.

---## 5. Leitura gerencial*Cinco linhas para quem não programa. Preencha depois de rodar os números acima.*1. De cada 100 alunos que chegaram ao fim do vínculo, ___ saíram sem concluir.2. O problema não está espalhado: ___ tem a pior taxa, bem acima da média da escola.3. ___ é o maior curso em evasões absolutas, mas em proporção está dentro do esperado — não é lá que se ganha mais.4. O turno da ___ concentra evasão acima do que o tamanho dele justifica.5. Recomendação: [ação concreta, com dono e prazo].